# Glue Interactive Sessions - Streaming Processing (CDC)

This notebook connects to an **AWS Glue interactive session** (account `123456789012`, region `us-east-1`) and runs the **same steps as the streaming mode** of the pipeline (`main._run_streaming`): reading via `reader.read(mode='streaming')` (Spark Structured Streaming), processing each micro-batch with quality validation, rejected-record writes and Delta MERGE, using `writeStream.foreachBatch`.

**Prerequisites:**

- Local **Glue PySpark** kernel: `pip install jupyter boto3 aws-glue-sessions` and then `install-glue-kernels`. Open with `jupyter notebook` and select the `Glue PySpark` kernel.
- Valid AWS credentials with Glue and S3 access (user `lake-admin` / group `datalake-admins`).
- Role `role-glue-job-flight-radar` with interactive session permissions (provisioned via Terraform in `infra/iam.tf`).
- `helpers.zip` published in the workspace bucket (`aws-glue/jobs/flight-radar/src/dependencies/helpers.zip`).

> **Delta Lake**: the `%%configure` cell below is required - the project's `writer.py` imports `delta.tables`. Without it the import fails with `ModuleNotFoundError: No module named 'delta'`.
>
> Adjust the role ARN and the bucket below if the AWS account differs from `331504768406`.
>
> The notebook uses a **test checkpoint** (suffix `interactive-test/`) to avoid conflicting with the production job `glue-flight-radar-streaming`. Prefer running it with the streaming job stopped.

In [ ]:
# 1) Session configuration (AWS Glue kernel magics)
# The next cell (%%configure) enables Delta Lake. Cell magics
# (%%configure/%%tags) must be the first line of the cell, with no comments.
%glue_version 5.0
%iam_role arn:aws:iam::331504768406:role/role-glue-job-flight-radar
%region us-east-1
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%session_id_prefix flight-radar-stream

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore"
}

In [ ]:
%%tags
{"Environment": "production", "Project": "flight-radar-glue", "Mode": "streaming"}

In [ ]:
# Project dependencies (helpers.zip already published in the workspace bucket)
%extra_py_files s3://lakehouse-workspace-331504768406/aws-glue/jobs/flight-radar/src/dependencies/helpers.zip

In [ ]:
# Instantiate the SparkSession / GlueContext
# In interactive sessions Spark already exists; getOrCreate returns the same session.
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

print('Spark version:', spark.version)

In [ ]:
# Load the configuration and choose the table to test
# Project modules loaded via %extra_py_files (helpers.zip) - `src` package.
import boto3
from src.dependencies.processor import Processor
from src.dependencies.config_models import Config

account_id = boto3.client('sts').get_caller_identity()['Account']
config = Config.from_s3(f's3://lakehouse-workspace-{account_id}/aws-glue/jobs/flight-radar/src/dependencies/config/config.json')

processor = Processor(spark)
source = config.get_source('aircraft')
target = source.target
print(source.source, '->', f'{target.database}.{target.table}')
print('CDC path:', source.cdc_source_location)

## Steps equivalent to `main._run_streaming`

`main._run_streaming` starts one Spark Structured Streaming query per table: it reads the CDC via `reader.read(mode='streaming')` and uses `foreachBatch` to validate, write rejected records and run the Delta MERGE in each micro-batch. Here we run the same sequence in separate cells.

To generate micro-batches during the test, **drop CDC Parquet files** (with `Op` and `dms_timestamp` columns) into the `CDC path` above.

In [ ]:
# Step 1 - Streaming read of the CDC (reader.read with mode='streaming')
# Reads from source.cdc_source_location with includeExistingFiles=false and
# cleanSource=archive (processed files go to *_archive/).
stream_df = processor._reader.read(source, mode='streaming')
print('isStreaming:', stream_df.isStreaming)

In [ ]:
# Step 2 - process_batch function (validation + rejects + Delta MERGE)
# Mirrors the process_batch closure in main._run_streaming.
def process_batch(df, batch_id, src=source, tgt=target):
    print(f'[{src.source}] batch_id={batch_id} rows={df.count()}')
    valid_df, rejects_df = processor._data_quality.validate(df, tgt, src)
    processor._writer.write_rejects(rejects_df, tgt)
    processor._writer.write(valid_df, tgt, src)
    print(f'[{src.source}] batch {batch_id} done')

In [ ]:
# Step 3 - Start the streaming query (foreachBatch + 30s trigger)
# Uses a test checkpoint to avoid conflicting with the production job.
test_checkpoint = source.checkpoint_location.rstrip('/') + '/interactive-test/'

query = (
    stream_df.writeStream
    .foreachBatch(process_batch)
    .outputMode('append')
    .trigger(processingTime='30 seconds')
    .option('checkpointLocation', test_checkpoint)
    .start()
)
print('Query started - id:', query.id)
print('Checkpoint:', test_checkpoint)

In [ ]:
# Step 4 - Observe the micro-batches for a few seconds and stop the query
# Drop CDC files into the path above during this interval to see the
# process_batch prints. The query is stopped at the end of the cell.
import time

print('Processing micro-batches for 60 seconds...')
time.sleep(60)
print('Status:', query.status)
query.stop()
print('Query stopped.')

In [ ]:
# Verification - read the final Delta table (via Glue Catalog)
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, f'{target.database}.{target.table}')
print('Rows in table:', dt.toDF().count())
dt.toDF().show(5)

In [ ]:
# Session status (shows tags, role, workers, region)
%status

In [ ]:
# Stop the session when done
%stop_session